<a href="https://colab.research.google.com/github/KimWonjong123/ui-prototyping-agent/blob/main/model-finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!git clone https://github.com/KimWonjong123/ui-prototyping-agent.git

Cloning into 'ui-prototyping-agent'...
remote: Enumerating objects: 56, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 56 (delta 19), reused 48 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (56/56), 43.87 KiB | 14.62 MiB/s, done.
Resolving deltas: 100% (19/19), done.


In [3]:
cd ui-prototyping-agent/

/content/ui-prototyping-agent


In [4]:
!pip install -r requirements.txt

In [5]:
!nvidia-smi

Thu May 14 04:31:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   29C    P0             47W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [8]:
# Unsloth 설치 (약 2-3분 소요)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-l074wn3z/unsloth_76d41af0e08c4864aebfc70b9cc753b8
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-l074wn3z/unsloth_76d41af0e08c4864aebfc70b9cc753b8
  Resolved https://github.com/unslothai/unsloth.git to commit 1c2a86f84a145e0e9e8a84409de542131036f857
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached xformers-0.0.26.post1.tar.gz (4.1 MB)
  Preparing metadata (setup.py) ... done
  Using cached trl-0.8.6-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.8.6-py3-none-any.whl (245 kB)


In [9]:
# Google Drive 마운트 (대용량 데이터 사용 시)
# from google.colab import drive
# drive.mount('/content/drive')

# 또는 직접 업로드
from google.colab import files
uploaded = files.upload()  # training_data.jsonl 선택

Saving training_data.jsonl to training_data.jsonl


In [10]:
# 데이터 확인
import json

TRAINING_DATA_PATH = "training_data.jsonl"  # 업로드한 파일명으로 변경

# 샘플 확인
with open(TRAINING_DATA_PATH, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        item = json.loads(line)
        print(f"=== Sample {i+1} ===")
        print(f"User: {item['messages'][1]['content'][:100]}...")
        print(f"Assistant: {item['messages'][2]['content'][:100]}...")
        print()

# 총 개수
with open(TRAINING_DATA_PATH, 'r', encoding='utf-8') as f:
    total = sum(1 for _ in f)
print(f"총 학습 데이터: {total}개")

=== Sample 1 ===
User: 활동자료실 관리 페이지 만들어줘. 검색 기능과 데이터 목록 표시가 있어야 해...
Assistant: <body>
<div id="wrap">
<header>
<div class="title">
<h1>활동자료실 관리(비결재)</h1>
</div>
<div class="search...

=== Sample 2 ===
User: 사용자가 자료를 검색하고 조회할 수 있는 화면이 필요해...
Assistant: <body>
<div id="wrap">
<header>
<div class="title">
<h1>활동자료실 관리(비결재)</h1>
</div>
<div class="search...

=== Sample 3 ===
User: 관리자가 활동자료를 등록하고 확인할 수 있는 전체 화면 만들어줘...
Assistant: <body>
<div id="wrap">
<header>
<div class="title">
<h1>활동자료실 관리(비결재)</h1>
</div>
<div class="search...

총 학습 데이터: 13275개


In [11]:
from unsloth import FastLanguageModel
import torch

# 설정
MAX_SEQ_LENGTH = 2048
MODEL_NAME = "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit"

# 모델 로드
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

print("✅ 모델 로드 완료!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

✅ 모델 로드 완료!


In [12]:
# LoRA 어댑터 추가
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

model.print_trainable_parameters()

Unsloth 2026.5.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


trainable params: 80,740,352 || all params: 7,696,356,864 || trainable%: 1.0491


In [13]:
from datasets import Dataset

# 데이터 로드
data = []
with open(TRAINING_DATA_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line.strip()))

print(f"데이터 로드: {len(data)}개")

# 포맷팅
def format_prompt(example):
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

# Dataset 생성
dataset = Dataset.from_list(data)
dataset = dataset.map(format_prompt, remove_columns=dataset.column_names)

# Train/Val 분리
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split['train']
eval_dataset = split['test']

print(f"✅ Train: {len(train_dataset)}, Validation: {len(eval_dataset)}")

데이터 로드: 13275개


Map:   0%|          | 0/13275 [00:00<?, ? examples/s]

✅ Train: 11947, Validation: 1328


In [14]:
from trl import SFTTrainer
from transformers import TrainingArguments

OUTPUT_DIR = "../drive/MyDrive/qwen-html-ui-lora"

# 학습 설정
# T4: batch_size=2, A100: batch_size=4
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,  # T4용, A100은 4
    gradient_accumulation_steps=16,   # 효과적 배치=16
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    bf16=True,
    optim="adamw_8bit",
    seed=42,
    report_to="none",
)

# Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    args=training_args,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/11947 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/1328 [00:00<?, ? examples/s]

In [16]:
# 학습 시작 (T4 기준 약 2-4시간 소요)
print("🚀 학습 시작...")
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🚀 학습 시작...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 11,947 | Num Epochs = 3 | Total steps = 1,122
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 8 x 1) = 32
 "-____-"     Trainable parameters = 80,740,352 of 7,696,356,864 (1.05% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,1.718531
20,1.157898
30,0.742058
40,0.579603
50,0.515382
60,0.472743
70,0.433602
80,0.424142
90,0.404454
100,0.356066


Unsloth: Restored added_tokens_decoder metadata in ./qwen-html-ui-lora/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./qwen-html-ui-lora/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./qwen-html-ui-lora/checkpoint-600/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./qwen-html-ui-lora/checkpoint-800/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./qwen-html-ui-lora/checkpoint-1000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./qwen-html-ui-lora/checkpoint-1122/tokenizer_config.json.


TrainOutput(global_step=1122, training_loss=0.16157301225817353, metrics={'train_runtime': 4656.3813, 'train_samples_per_second': 7.697, 'train_steps_per_second': 0.241, 'total_flos': 1.1014030010531359e+18, 'train_loss': 0.16157301225817353, 'epoch': 3.0})

In [17]:
# 모델 저장
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ 모델 저장 완료: {OUTPUT_DIR}")

Unsloth: Restored added_tokens_decoder metadata in ../drive/MyDrive/qwen-html-ui-lora/tokenizer_config.json.


✅ 모델 저장 완료: ../drive/MyDrive/qwen-html-ui-lora


In [18]:
# 추론 모드로 전환
FastLanguageModel.for_inference(model)

# 테스트 프롬프트
test_prompts = [
    "로그인 페이지 만들어줘",
    "검색 기능이 있는 헤더 영역 필요해",
    "저장 버튼과 닫기 버튼이 있는 하단 영역 만들어줘",
]

system_prompt = """당신은 HTML/CSS UI 전문가입니다. 사용자의 요청에 따라 기존 디자인 시스템과 일관된 HTML 코드를 생성합니다.
- 시맨틱 HTML5 태그를 사용합니다
- 클래스명은 기존 컴포넌트 라이브러리의 규칙을 따릅니다
- 접근성(a11y)을 고려합니다
- 깔끔하고 유지보수하기 쉬운 코드를 작성합니다"""

for prompt in test_prompts:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        inputs,
        max_new_tokens=1024,
        temperature=0.7,
        do_sample=True,
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print(f"\n{'='*60}")
    print(f"📝 요청: {prompt}")
    print(f"{'='*60}")
    print(response.split("assistant")[-1].strip())

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn


📝 요청: 로그인 페이지 만들어줘
<body>
<div class="login">
<h2 class="hide">로그인</h2>
<div class="guidance-message-pnl">
<p>한화손보 Mall로 이동합니다.</p>
</div>
<div class="buttonset">
<a class="btn_cont" href="#none" id="moveHncMall">활동센터 바로가기</a>
</div>
<div class="extra-dimensions" id="areaHdn"></div>
</div>
</body>


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📝 요청: 검색 기능이 있는 헤더 영역 필요해
<header>
<div class="title">
<h1>활동크레딧 당월 지급완료 상세 내역</h1>
</div>
<div class="search" tabindex="0">
<table id="tblSearch">
<colgroup>
<col width="6.0%"/>
<col width="94.0%"/>
</colgroup>
<tbody>
<tr data-url="../cm/sfacmz05.div" id="sfacmz05div"></tr>
<tr>
<th>조회조건</th>
<td>
<!-- inqSrhcd :: 기준년월 --><span class="selection-grp"><input checked="" id="rdoSrhgcd1" name="inqSrhcd" type="radio" value="1"/><label for="rdoSrhgcd1">기준년월</label></span> <input data-nx="{mask:'yyyy-MM-dd',type:'date',calendar:true}" id="txtSrhRcpStrDt" size="8" type="text"/> ~ <input data-nx="{mask:'yyyy-MM-dd',type:'date',calendar:true}" id="txtSrhRcpNdDt" size="8" type="text"/>  
 <!-- inqSrhcd :: 개별조회 --><span class="selection-grp"><input id="rdoSrhcd2" name="inqSrhcd" type="radio" value="2"/><label for="rdoSrhcd2">개별조회</label></span> <span class="combobox-pnl" data-prop="{valueCol:'valueCol',textCol:'textCol',bizcodeRef:'dsComboNdvInqFlg',width:'100',value:'0'}" data-widget="combobox"

In [21]:
# GGUF로 변환 (Q4_K_M 양자화)
# 로컬 CPU에서 실행하기 위한 형식

GGUF_DIR = "./qwen-html-ui-gguf"

model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method="q4_k_m"  # 4bit 양자화
)

print(f"✅ GGUF 파일 생성: {GGUF_DIR}/unsloth.Q4_K_M.gguf")

Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in ./qwen-html-ui-gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...


Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [00:48<00:00, 12.21s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [01:02<00:00, 15.74s/it]


Unsloth: Merge process complete. Saved to `/content/ui-prototyping-agent/qwen-html-ui-gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['./qwen-html-ui-gguf_gguf/qwen2.5-coder-7b-instruct.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['./qwen-html-ui-gguf_gguf/qwen2.5-coder-7b-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model ./qwen-html-ui-gguf_gguf/qwen2.5-coder-7b-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to ./qwen-html-ui-gguf_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f ./qwen-html-ui-gguf_gguf/Modelfile
✅ GGUF 파일 생성: ./qwen-html-ui-gguf/unsloth.Q4_K_M.gguf


In [22]:
# Ollama용 Modelfile 생성
modelfile = '''FROM ./unsloth.Q4_K_M.gguf

TEMPLATE """{{ if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}<|im_start|>user
{{ .Prompt }}<|im_end|>
<|im_start|>assistant
"""

PARAMETER stop "<|im_end|>"
PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER num_ctx 2048

SYSTEM """당신은 HTML/CSS UI 전문가입니다. 사용자의 요청에 따라 기존 디자인 시스템과 일관된 HTML 코드를 생성합니다.
- 시맨틱 HTML5 태그를 사용합니다
- 클래스명은 기존 컴포넌트 라이브러리의 규칙을 따릅니다
- 접근성(a11y)을 고려합니다
- 깔끔하고 유지보수하기 쉬운 코드를 작성합니다"""
'''

with open(f"{GGUF_DIR}/Modelfile", "w") as f:
    f.write(modelfile)

print("✅ Modelfile 생성 완료!")

✅ Modelfile 생성 완료!


In [28]:
!rm -rf ../drive/MyDrive/qwen-html-ui-lora/*
!rm -rf ../drive/MyDrive/qwen-html-ui-gguf_gguf/*
!rm -rf ../drive/MyDrive/qwen-html-ui-gguf/*

In [29]:
!cp -r ./qwen-html-ui-lora/* ../drive/MyDrive/qwen-html-ui-lora/
!cp -r ./qwen-html-ui-gguf_gguf/* ../drive/MyDrive/qwen-html-ui-gguf_gguf/
!cp -r ./qwen-html-ui-gguf/* ../drive/MyDrive/qwen-html-ui-gguf/